El **VIF** se trata del **Virus de Inmunodeficiencia Felina**, el cual, es un
Lentivirus de la familia de los retrovirus que genera el **SIDA felino**. Aunque existe una vacuna tradicional basada en virus atenuados, su eficacia y utilidad clínica siguen siendo inciertas. Por ello, resulta de gran interés el desarrollo de una vacuna de ARNm para esta enfermedad, que afecta aproximadamente al 11% de la población felina mundial, siendo las colonias de gatos callejeros el principal grupo de riesgo.

El primer paso consiste en seleccionar una proteína diana que actúe como antígeno en el marco de lectura abierto (ORF) de la vacuna. En este trabajo se ha optado por la glicoproteína de envoltura Env, ya que es la estructura clave que facilita la entrada del virus a los linfocitos T colaboradores (T helper) del felino.

*Kenyon, J. C., & Lever, A. M. (2011). The molecular biology of feline immunodeficiency virus (FIV). Viruses, 3(11), 2192–2213. https://doi.org/10.3390/v3112192*

1) **Obtención de la secuencia de referencia**: Extraemos la secuencia de ADN codificante para el gen de la proteína Env del VIF (Access ID: A0A0D5ZXZ6).

In [14]:
# Cargar herramienta de Biopython
from Bio import SeqIO
from Bio.Seq import Seq

# Creamos la secuencia a partir del archivo fasta
cds_env = SeqIO.read("env_gene_felis_catus.fasta","fasta").seq

# Ahora transcribimos la secuencia
orf_env = cds_env.transcribe()
print(orf_env)


AUGAAACUCCCAACAGGAAUGGUCAUUUUAUGUAGCCUAAUAAUAGUUCGGGCAGGGUUUGACGACCCCCGCAAGGCUAUCGCAUUAGUACAAAAACAACAUGGUAAACCAUGCGAAUGCAGCGGAGGGCAGGUAUCCGAGGCCCCACCGAACUCCAUCCAACAGGUAACUUGCCCAGGCAAGACGGCCUACUUAAUGACCAACCAAAAAUGGAAAUGCAGAGUCACUCCAAAAAAUCUCACCCCUAGCGGGGGAGAACUCCAGAACUGCCCCUGUAACACUUUCCAGGACUCGAUGCACAGUUCUUGUUAUACUGAAUACCGGCAAUGCAGGGCGAAUAAUAAGACAUACUACACGGCCACCUUGCUUAAAAUACGGUCUGGGAGCCUCAACGAGGUACAGAUAUUACAAAACCCCAAUCAGCUCCUACAGUCCCCUUGUAGGGGCUCUAUAAAUCAGCCCGUUUGCUGGAGUGCCACAGCCCCCAUCCAUAUCUCCGAUGGUGGAGGACCCCUCGAUACUAAGAGAGUGUGGACAGUCCAAAAAAGGCUAGAACAAAUUCAUAAGGCUAUGCAUCCUGAACUUCAAUACCACCCCUUAGCCCUGCCCAAAGUCAGAGAUGACCUUAGCCUUGAUGCACGGACUUUUGAUAUCCUGAAUACCACUUUUAGGUUACUCCAGAUGUCCAAUUUUAGCCUUGCCCAAGAUUGUUGGCUCUGUUUAAAACUAGGUACCCCUACCCCUCUUGCGAUACCCACUCCCUCUUUAACCUACUCCCUAGCAGACUCCCUAGCGAAUGCCUCCUGUCAGAUUAUACCUCCCCUCUUGGUUCAACCGAUGCAGUUAUCCAACUCGUCCUGUUUAUCUUCCCCUUUCAUUAACGAUACGGAACAAAUAGACUUAGGUGCAGUCACCUUUACUAACUGCACCUCUGUAGCCAAUGUCAGUAGUCCUUUAUGUGCCCUAAACGGGUCAGUCUUCCUCUGUGGAAAUAACAUGG

2) **Optimización de codones**: Adaptamos la secuencia al sesgo de uso de codones de los ARNt disponibles en _Felis catus_. No se han seleccionado sistemáticamente los codones de máxima eficiencia, ya que una tasa de traducción excesivamente rápida podría impedir las pausas ribosómicas necesarias para el correcto plegamiento de la proteína en la membrana.

In [ ]:
# Creamos una función encargada de cambiar los codones para su optimización
def felis_codon_optimizer(sequence):

    # Creamos un diccionario con los codondes frente a los codones óptimos (no con la máxima eficiencia en todo caso)
    codon_optimized = {
    "AAA": "AAG", "AAC": "AAC", "AAG": "AAG", "AAU": "AAC",
    "ACA": "ACC", "ACC": "ACC", "ACG": "ACC", "ACU": "ACC",
    "AGA": "CGG", "AGC": "UCC", "AGG": "CGG", "AGU": "UCC",
    "AUA": "AUC", "AUC": "AUC", "AUG": "AUG", "AUU": "AUC",
    
    "CAA": "CAG", "CAC": "CAC", "CAG": "CAG", "CAU": "CAC",
    "CCA": "CCC", "CCC": "CCC", "CCG": "CCC", "CCU": "CCC",
    "CGA": "CGG", "CGC": "CGG", "CGG": "CGG", "CGU": "CGG",
    "CUA": "CUG", "CUC": "CUG", "CUG": "CUG", "CUU": "CUG",
    
    "GAA": "GAG", "GAC": "GAC", "GAG": "GAG", "GAU": "GAC",
    "GCA": "GCC", "GCC": "GCC", "GCG": "GCC", "GCU": "GCC",
    "GGA": "GGC", "GGC": "GGC", "GGG": "GGC", "GGU": "GGC",
    "GUA": "GUG", "GUC": "GUG", "GUG": "GUG", "GUU": "GUG",
    
    "UAA": "UGA", "UAC": "UAC", "UAG": "UGA", "UAU": "UAC",
    "UCA": "UCC", "UCC": "UCC", "UCG": "UCC", "UCU": "UCC",
    "UGA": "UGA", "UGC": "UGC", "UGG": "UGG", "UGU": "UGC",
    "UUA": "CUG", "UUC": "UUC", "UUG": "CUG", "UUU": "UUC"
    }
    
    
    optimized_seq = ""
    
    # Ahora optimizamos los codones empleando el diccionario anterior
    for i in range(0,len(sequence),3):
        codon = sequence[i:i+3]
        optimized_seq += codon_optimized.get(codon,"")
    
    return optimized_seq

optimized_orf_env = felis_codon_optimizer(orf_env)
        


3) **Incorporación de regiones 5'-UTR y 3'-UTR**: Estas secuencias se obtienen del gen endógeno de la Caveolina-1 de _Felis catus_ (ID: A0M8S7), al tratarse de un gen altamente conservado que favorece la estabilidad y expresión del transcrito. En concreto, se utilizan los 57 nucleótidos previos a la secuencia codificante (52425...52481) para la región 5'-UTR y los 116 nucleótidos posteriores (98057...98173) para la 3'-UTR.

In [ ]:
# Creamos las regiones UTR incluidas en el archivo fasta
utr_regions = SeqIO.parse("utr_regions_cav_1.fasta","fasta")

# Diferenciamos entre región 5'-UTR y 3'-UTR, y transcribimos
leader_region = next(utr_regions).seq.transcribe()
trailer_region = next(utr_regions).seq.transcribe()

print(leader_region)
print(trailer_region)


AGGCGAUCCUGCGAGUGAGAUCCCGGACAGGAGGAGGACCUCCUUUGACACACAAACA
AGAGGAAAGAACCAAGCAAUAUUGAGCCAUAGCUAUCCUAAGUGGUCUGCAUUUCUGCUGUAAAAUGCAAUUUGGAAAAAAUAAACGGAAAAAAAAAGGAACUGUAAAGGAAACCAG


4) **Ensamblaje de la estructura**: Ensamblamos la secuencia pre-ARNm completa, incorporando la caperuza 5' (capping) y la cola poli-A en el extremo 3'.

In [17]:
def mRNA_sequence_junction(sequence,utr_5,utr_3):
    poli_a_tail = ("A" * 10) + "GCAUAUGACU" + ("A" * 70)
    
    proto_mRNA_vaccine = "mG-" + utr_5 + sequence + utr_3 + poli_a_tail
    
    return proto_mRNA_vaccine

proto_mRNA_seq = mRNA_sequence_junction(optimized_orf_env,leader_region,trailer_region)
print(proto_mRNA_seq)

mG-AGGCGAUCCUGCGAGUGAGAUCCCGGACAGGAGGAGGACCUCCUUUGACACACAAACAAUGAAGCUGCCCACCGGCAUGGUGAUCCUGUGCUCCCUGAUCAUCGUGCGGGCCGGCUUCGACGACCCCCGGAAGGCCAUCGCCCUGGUGCAGAAGCAGCACGGCAAGCCCUGCGAGUGCUCCGGCGGCCAGGUGUCCGAGGCCCCCCCCAACUCCAUCCAGCAGGUGACCUGCCCCGGCAAGACCGCCUACCUGAUGACCAACCAGAAGUGGAAGUGCCGGGUGACCCCCAAGAACCUGACCCCCUCCGGCGGCGAGCUGCAGAACUGCCCCUGCAACACCUUCCAGGACUCCAUGCACUCCUCCUGCUACACCGAGUACCGGCAGUGCCGGGCCAACAACAAGACCUACUACACCGCCACCCUGCUGAAGAUCCGGUCCGGCUCCCUGAACGAGGUGCAGAUCCUGCAGAACCCCAACCAGCUGCUGCAGUCCCCCUGCCGGGGCUCCAUCAACCAGCCCGUGUGCUGGUCCGCCACCGCCCCCAUCCACAUCUCCGACGGCGGCGGCCCCCUGGACACCAAGCGGGUGUGGACCGUGCAGAAGCGGCUGGAGCAGAUCCACAAGGCCAUGCACCCCGAGCUGCAGUACCACCCCCUGGCCCUGCCCAAGGUGCGGGACGACCUGUCCCUGGACGCCCGGACCUUCGACAUCCUGAACACCACCUUCCGGCUGCUGCAGAUGUCCAACUUCUCCCUGGCCCAGGACUGCUGGCUGUGCCUGAAGCUGGGCACCCCCACCCCCCUGGCCAUCCCCACCCCCUCCCUGACCUACUCCCUGGCCGACUCCCUGGCCAACGCCUCCUGCCAGAUCAUCCCCCCCCUGCUGGUGCAGCCCAUGCAGCUGUCCAACUCCUCCUGCCUGUCCUCCCCCUUCAUCAACGACACCGAGCAGAUCGACCUGGGCGCCGUGACCUUCACCAACUGCACCUCCGUGGCC

5) **Modificación química**: Para incrementar la estabilidad y evitar la degradación por parte del sistema inmunitario innato del hospedador, sustituimos todos los uracilos por pseudouridinas ($\psi$).

In [18]:
def pseudouridination(mRNA_seq):
    mRNA_vaccine = str(mRNA_seq).replace("U","ψ")
    return mRNA_vaccine

print(pseudouridination(proto_mRNA_seq))

mG-AGGCGAψCCψGCGAGψGAGAψCCCGGACAGGAGGAGGACCψCCψψψGACACACAAACAAψGAAGCψGCCCACCGGCAψGGψGAψCCψGψGCψCCCψGAψCAψCGψGCGGGCCGGCψψCGACGACCCCCGGAAGGCCAψCGCCCψGGψGCAGAAGCAGCACGGCAAGCCCψGCGAGψGCψCCGGCGGCCAGGψGψCCGAGGCCCCCCCCAACψCCAψCCAGCAGGψGACCψGCCCCGGCAAGACCGCCψACCψGAψGACCAACCAGAAGψGGAAGψGCCGGGψGACCCCCAAGAACCψGACCCCCψCCGGCGGCGAGCψGCAGAACψGCCCCψGCAACACCψψCCAGGACψCCAψGCACψCCψCCψGCψACACCGAGψACCGGCAGψGCCGGGCCAACAACAAGACCψACψACACCGCCACCCψGCψGAAGAψCCGGψCCGGCψCCCψGAACGAGGψGCAGAψCCψGCAGAACCCCAACCAGCψGCψGCAGψCCCCCψGCCGGGGCψCCAψCAACCAGCCCGψGψGCψGGψCCGCCACCGCCCCCAψCCACAψCψCCGACGGCGGCGGCCCCCψGGACACCAAGCGGGψGψGGACCGψGCAGAAGCGGCψGGAGCAGAψCCACAAGGCCAψGCACCCCGAGCψGCAGψACCACCCCCψGGCCCψGCCCAAGGψGCGGGACGACCψGψCCCψGGACGCCCGGACCψψCGACAψCCψGAACACCACCψψCCGGCψGCψGCAGAψGψCCAACψψCψCCCψGGCCCAGGACψGCψGGCψGψGCCψGAAGCψGGGCACCCCCACCCCCCψGGCCAψCCCCACCCCCψCCCψGACCψACψCCCψGGCCGACψCCCψGGCCAACGCCψCCψGCCAGAψCAψCCCCCCCCψGCψGGψGCAGCCCAψGCAGCψGψCCAACψCCψCCψGCCψGψCCψCCCCCψψCAψCAACGACACCGAGCAGAψCGACCψGGGCGCCGψGACCψψCACCAACψGCACCψCCGψGGCC

6) **Integración**: Definimos una función única que agrupa y automatiza todas las etapas del diseño del ARNm.

In [ ]:
# Cargar herramienta de Biopython
from Bio import SeqIO

# Creamos la secuencia a partir del archivo fasta
cds_env = SeqIO.read("env_gene_felis_catus.fasta","fasta").seq

# Creamos las regiones UTR incluidas en el archivo fasta
utr_regions = SeqIO.parse("utr_regions_cav_1.fasta","fasta")

# Diferenciamos entre región 5'-UTR y 3'-UTR y transcribimos
leader_region = next(utr_regions).seq.transcribe()
trailer_region = next(utr_regions).seq.transcribe()

# Creamos la función que agrupe todo lo visto previamente
def mRNA_vaccine_generator(cds_sequence,utr_5,utr_3):

    # Ahora transcribimos las secuencias
    orf_seq = cds_sequence.transcribe()

    optimized_seq = felis_codon_optimizer(orf_seq)
    
    pre_mRNA_vaccine = mRNA_sequence_junction(optimized_seq,utr_5,utr_3)
    
    mRNA_FIV_vaccine = pseudouridination(pre_mRNA_vaccine)
    
    return mRNA_FIV_vaccine
    
mRNA_vaccine_generator(cds_env,leader_region,trailer_region)

'mG-AGGCGAψCCψGCGAGψGAGAψCCCGGACAGGAGGAGGACCψCCψψψGACACACAAACAAψGAAGCψGCCCACCGGCAψGGψGAψCCψGψGCψCCCψGAψCAψCGψGCGGGCCGGCψψCGACGACCCCCGGAAGGCCAψCGCCCψGGψGCAGAAGCAGCACGGCAAGCCCψGCGAGψGCψCCGGCGGCCAGGψGψCCGAGGCCCCCCCCAACψCCAψCCAGCAGGψGACCψGCCCCGGCAAGACCGCCψACCψGAψGACCAACCAGAAGψGGAAGψGCCGGGψGACCCCCAAGAACCψGACCCCCψCCGGCGGCGAGCψGCAGAACψGCCCCψGCAACACCψψCCAGGACψCCAψGCACψCCψCCψGCψACACCGAGψACCGGCAGψGCCGGGCCAACAACAAGACCψACψACACCGCCACCCψGCψGAAGAψCCGGψCCGGCψCCCψGAACGAGGψGCAGAψCCψGCAGAACCCCAACCAGCψGCψGCAGψCCCCCψGCCGGGGCψCCAψCAACCAGCCCGψGψGCψGGψCCGCCACCGCCCCCAψCCACAψCψCCGACGGCGGCGGCCCCCψGGACACCAAGCGGGψGψGGACCGψGCAGAAGCGGCψGGAGCAGAψCCACAAGGCCAψGCACCCCGAGCψGCAGψACCACCCCCψGGCCCψGCCCAAGGψGCGGGACGACCψGψCCCψGGACGCCCGGACCψψCGACAψCCψGAACACCACCψψCCGGCψGCψGCAGAψGψCCAACψψCψCCCψGGCCCAGGACψGCψGGCψGψGCCψGAAGCψGGGCACCCCCACCCCCCψGGCCAψCCCCACCCCCψCCCψGACCψACψCCCψGGCCGACψCCCψGGCCAACGCCψCCψGCCAGAψCAψCCCCCCCCψGCψGGψGCAGCCCAψGCAGCψGψCCAACψCCψCCψGCCψGψCCψCCCCCψψCAψCAACGACACCGAGCAGAψCGACCψGGGCGCCGψGACCψψCACCAACψGCACCψCCGψGGC

7. **Estabilidad estructural (Control de Calidad 1)**: Para confirmar la viabilidad del transcrito, evaluamos su fracción GC, estableciendo un criterio mínimo de viabilidad del 40%.

In [20]:
from Bio.SeqUtils import gc_fraction

gc_fract = gc_fraction(optimized_orf_env)
print(f"La fracción GC es del {gc_fract:.2f}.")

if gc_fract > 0.4:
    print("¡Tiene una fracción GC óptima!")
else:
    print("¡Cuidado! ¡Baja estabilidad por baja fración GC!")

La fracción GC es del 0.67.
¡Tiene una fracción GC óptima!


8. **Validación de integridad del antígeno (Control de Calidad 2)**: Por último, para asegurar que la modificación no ha alterado el marco de lectura, traducimos el ORF optimizado y comprobamos si la secuencia de aminoácidos resultante codifica exactamente para la proteína Env original del virus.

In [ ]:
# Traducimos ambas secuencias
original_prot = str(orf_env.translate())
optimized_seq_prot = str(Seq(optimized_orf_env).translate())


if original_prot == optimized_seq_prot:
    print("¡Las secuencias coinciden!")
else:
    print("¡Cuidado! ¡Las secuencias no coinciden!\n\nLas no coincidencias son las siguientes:\n")
    
    for i, (a,b) in enumerate(zip(original_prot,optimized_seq_prot)):
        if a != b:
            print(f"{a}{i}{b}")
    

¡Las secuencias coinciden!
